In [6]:
%pip install -U transformers tokenizers huggingface_hub sentencepiece tiktoken

Note: you may need to restart the kernel to use updated packages.


In [7]:
import torch
import pickle

# load .pt file
data = torch.load("pair_embeddings.pt", map_location="cpu")

# save as .pkl
with open("pair_embeddings.pkl", "wb") as f:
    pickle.dump(data, f)

print("Converted to .pkl successfully")

C:\Users\mg276\AppData\Local\Temp\ipykernel_4128\3793057122.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load("pair_embeddings.pt", map_location="cpu")


Converted to .pkl successfully


In [ ]:
F:\\SIN\\Dataset\\final_processed_data.csv

In [9]:
import pandas as pd
import torch
from torch_geometric.data import Data
from transformers import AutoTokenizer, AutoModel, BertTokenizer, BertTokenizerFast
from tqdm import tqdm

# =========================
# CONFIG
# =========================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PAIR_CSV = "F:\\SIN\\Dataset\\final_processed_data.csv"
COL1 = "SMILES1"
COL2 = "SMILES2"
TARGET_DIM = 256
BIOBERT = "monologg/biobert_v1.1_pubmed"

# =========================
# LOAD PAIR EMBEDDINGS + LABELS
# =========================
bundle = torch.load("pair_embeddings.pt", map_location="cpu")

pair_features = bundle["X"].float()
Y = bundle["y"].float()
label_texts = [str(x) for x in bundle["side_effect_classes"]]

if Y.shape[1] != len(label_texts):
    raise ValueError(
        f"Label count mismatch: y has {Y.shape[1]} columns, "
        f"but side_effect_classes has {len(label_texts)} entries"
    )

print("Loaded pair features:", pair_features.shape)
print("Loaded pair labels:", Y.shape)
print("Loaded side-effect label texts:", len(label_texts))

# =========================
# OPTIONAL PAIR KEYS (for metadata/debugging)
# =========================
df = pd.read_csv(PAIR_CSV)
if len(df) != pair_features.shape[0]:
    print(
        "Warning: row count in CSV differs from X rows "
        f"({len(df)} vs {pair_features.shape[0]})."
    )

def canonical_pair(a, b):
    return tuple(sorted([a, b]))

pair_keys = [canonical_pair(a, b) for a, b in zip(df[COL1], df[COL2])]

# =========================
# BIOBERT ENCODING FOR SIDE-EFFECT NODES
# =========================
print("Loading BioBERT on", DEVICE)

# Some environments have an AutoTokenizer mapping/backend issue.
# Load explicit BERT tokenizer classes first, then fallback.
try:
    tokenizer = BertTokenizerFast.from_pretrained(BIOBERT)
    print("Using BertTokenizerFast")
except Exception:
    try:
        tokenizer = BertTokenizer.from_pretrained(BIOBERT)
        print("Using BertTokenizer (slow)")
    except Exception:
        tokenizer = AutoTokenizer.from_pretrained(BIOBERT, use_fast=False)
        print("Using AutoTokenizer(use_fast=False) fallback")

model = AutoModel.from_pretrained(BIOBERT).to(DEVICE)
model.eval()


def encode_labels(texts, batch_size=32):
    chunks = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding label text"):
        batch = texts[i : i + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=32,
        ).to(DEVICE)

        with torch.no_grad():
            out = model(**inputs)

        cls = out.last_hidden_state[:, 0, :]
        chunks.append(cls.cpu())

    return torch.cat(chunks, dim=0)


label_features = encode_labels(label_texts)
print("BioBERT label features:", label_features.shape)

# =========================
# PROJECT TO SHARED DIMENSION
# =========================
proj_pair = torch.nn.Linear(pair_features.shape[1], TARGET_DIM)
proj_label = torch.nn.Linear(label_features.shape[1], TARGET_DIM)

pair_features = proj_pair(pair_features)
label_features = proj_label(label_features)

print("Projected pair features:", pair_features.shape)
print("Projected label features:", label_features.shape)

# =========================
# BUILD BIPARTITE EDGES: pair <-> side_effect
# =========================
num_pairs = pair_features.shape[0]
num_labels = label_features.shape[0]

edge_src = []
edge_dst = []

Y_bin = (Y > 0).long()
for p_idx in range(num_pairs):
    pos_labels = torch.where(Y_bin[p_idx] == 1)[0]
    for l in pos_labels.tolist():
        label_node = num_pairs + l

        # pair -> label
        edge_src.append(p_idx)
        edge_dst.append(label_node)

        # label -> pair (undirected via two directed edges)
        edge_src.append(label_node)
        edge_dst.append(p_idx)

edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
print("edge_index shape:", edge_index.shape)

# =========================
# FINAL GRAPH OBJECT
# =========================
x = torch.cat([pair_features, label_features], dim=0)

graph_data = Data(x=x, edge_index=edge_index)

# metadata useful for downstream link prediction
graph_data.num_pairs = num_pairs
graph_data.num_labels = num_labels
graph_data.pair_keys = pair_keys
graph_data.side_effect_text = label_texts

# =========================
# SAVE ARTIFACTS
# =========================
torch.save(graph_data, "graph_data.pt")
torch.save(Y_bin, "Y_pair.pt")

torch.save(
    {
        "pair_features": pair_features,
        "label_features": label_features,
        "Y": Y_bin,
        "pair_keys": pair_keys,
        "side_effect_text": label_texts,
    },
    "bipartite_graph_bundle.pt",
)

print("✅ DONE: Built bipartite graph with BioBERT label embeddings")

C:\Users\mg276\AppData\Local\Temp\ipykernel_5052\695262592.py:20: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  bundle = torch.load("pair_embeddings.pt", map_location="cpu")

Loaded pair features: torch.Size([63304, 256])
Loaded pair labels: torch.Size([63304, 963])
Loaded side-effect label texts: 963
Loading BioBERT on cuda
Using BertTokenizerFast


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11255.55it/s]
BertModel LOAD REPORT from: monologg/biobert_v1.1_pubmed
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Encoding label text: 100%|██████████| 31/31 [00:00<00:00, 137.27it/s]


BioBERT label features: torch.Size([963, 768])
Projected pair features: torch.Size([63304, 256])
Projected label features: torch.Size([963, 256])
edge_index shape: torch.Size([2, 9120668])
✅ DONE: Built bipartite graph with BioBERT label embeddings
